# BIDS Conversion Pipeline for Neuroimaging Volumetric Data

**Authors:** Gal Gvili & Tamir Rahamim  

**Date:** January 2026  

**Purpose:** Convert CSV-formatted brain volumetric measurements to BIDS-compliant derivative format

---

## Overview

This notebook demonstrates a reproducible pipeline for converting neuroimaging data from lab-specific CSV format into the Brain Imaging Data Structure (BIDS) standard for sharing via OpenNeuro or similar repositories.

### What is BIDS?

BIDS (Brain Imaging Data Structure) is a community-developed standard for organizing and describing neuroimaging data. It ensures:

- **Consistency** across datasets
- **Machine-readability** for automated processing
- **Compatibility** with analysis tools
- **Ease of sharing** through repositories like OpenNeuro

### Our Data

- **Type:** Brain volumetric measurements (derivatives)
- **Source:** Structural MRI with Brainnetome Atlas (BNA) parcellation
- **Regions:** 8 bilateral brain regions (prefrontal cortex, amygdala, hippocampus)
- **Measurements:** Gray matter, white matter, and CSF volumes per region
- **Subjects:** 3 participants with pre/post sessions

## Step 1: Import Required Libraries

In [39]:
import pandas as pd
import json
import os
from pathlib import Path
import logging

# Configure logging for better output tracking
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s: %(message)s'
)
logger = logging.getLogger(__name__)

# Constants for brain regions and tissue types
BRAIN_REGIONS = ['A8m', 'A9l', 'A6dl', 'A10m', 'mAmyg', 'lAmyg', 'rHipp', 'cHipp']
HEMISPHERES = ['L', 'R']
TISSUE_TYPES = ['Vgm', 'Vwm', 'Vcsf']

print("✓ Libraries imported successfully")
print(f"✓ Brain regions defined: {len(BRAIN_REGIONS)} regions")

✓ Libraries imported successfully
✓ Brain regions defined: 8 regions


## Step 2: Load and Inspect Original Data

First, let's examine the structure of our CSV data files.

In [40]:
# Load the neuroimaging volumetric data
neuroimaging_df = pd.read_csv('Data/sample_data.csv')
print("Neuroimaging Data Shape:", neuroimaging_df.shape)
print("\nFirst few rows:")
display(neuroimaging_df.head())

print("\nColumn names (first 10):")
print(neuroimaging_df.columns[:10].tolist())

print("\nGroup distribution:")
print(neuroimaging_df['group'].value_counts())

Neuroimaging Data Shape: (3, 56)

First few rows:


,subject num,exp,label,brain bank,group,design,time,additional info,A8m_L Vgm,A8m_L Vwm,...,rHipp_L Vcsf,rHipp_R Vgm,rHipp_R Vwm,rHipp_R Vcsf,cHipp_L Vgm,cHipp_L Vwm,cHipp_L Vcsf,cHipp_R Vgm,cHipp_R Vwm,cHipp_R Vcsf
0,1001,novice,control,False,control,baseline,pre,NaN,2.82,0.98,...,0.36,1.36,0.55,0.49,0.89,0.50,0.08,1.13,0.64,0.16
1,1002,expert,patient,False,patient,baseline,pre,NaN,2.24,0.81,...,0.86,1.02,0.32,0.85,0.63,0.00,0.17,0.87,0.01,0.21
2,1003,novice,control,False,control,baseline,post,NaN,2.95,1.12,...,0.39,1.45,0.59,0.53,0.96,0.54,0.09,1.21,0.69,0.18



Column names (first 10):
['subject num', 'exp', 'label', 'brain bank', 'group', 'design', 'time', 'additional info', 'A8m_L Vgm', 'A8m_L Vwm']

Group distribution:
group
control    2
patient    1
Name: count, dtype: int64


In [41]:
# Load the questionnaire metadata
questionnaire_df = pd.read_csv('Data/questionnaire_metadata.csv')
print("Questionnaire Metadata Shape:", questionnaire_df.shape)
print("\nMetadata columns:")
display(questionnaire_df.head())

Questionnaire Metadata Shape: (3, 5)

Metadata columns:


,subject num,is_female,is_right_handed,age,additional_languages
0,1001,0,1,28,"English, Hebrew"
1,1002,1,1,34,"English, Russian"
2,1003,0,1,25,English


## Step 3: Define Helper Functions

These functions will help us convert the data to BIDS format with proper error handling.

In [42]:
def create_directory_structure(base_path, subject_id, session_id):
    """
    Create BIDS-compliant directory structure for derivatives.
    
    Parameters:
    -----------
    base_path : str
        Base directory for the BIDS dataset
    subject_id : str
        Subject identifier (e.g., 'sub-1001')
    session_id : str
        Session identifier (e.g., 'ses-pre')
    
    Returns:
    --------
    Path
        Path to the subject's anatomical directory
    """
    try:
        anat_dir = Path(base_path) / 'derivatives' / 'bna_volumes' / subject_id / session_id / 'anat'
        anat_dir.mkdir(parents=True, exist_ok=True)
        return anat_dir
    except OSError as e:
        logger.error(f"Failed to create directory {anat_dir}: {e}")
        raise

print("✓ Directory creation function defined")

✓ Directory creation function defined


In [43]:
def convert_csv_to_bids_tsv(input_csv, output_dir, subject_id, session_id):
    """
    Convert neuroimaging CSV data to BIDS derivative TSV format.
    
    This function:
    1. Reads the wide-format CSV file
    2. Filters data for specific subject and session
    3. Transforms from wide to long format
    4. Creates BIDS-compliant TSV file
    
    Parameters:
    -----------
    input_csv : str
        Path to input CSV file with brain volume data
    output_dir : str
        Output directory for BIDS dataset
    subject_id : str
        Subject identifier
    session_id : str
        Session identifier
    
    Returns:
    --------
    str or None
        Path to created TSV file, or None if conversion failed
    """
    # Validate input file exists
    input_path = Path(input_csv)
    if not input_path.exists():
        logger.error(f"Input file not found: {input_csv}")
        return None
    
    try:
        # Read input CSV
        df = pd.read_csv(input_csv)
    except pd.errors.EmptyDataError:
        logger.error(f"Input file is empty: {input_csv}")
        return None
    except Exception as e:
        logger.error(f"Failed to read CSV file: {e}")
        return None
    
    # Validate required columns
    required_cols = ['subject num', 'time']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        logger.error(f"Missing required columns: {missing_cols}")
        return None
    
    # Filter data for specific subject and session
    subject_num = int(subject_id.replace('sub-', ''))
    subject_data = df[df['subject num'] == subject_num]
    
    if subject_data.empty:
        logger.warning(f"No data found for subject {subject_id}")
        return None
    
    # Determine session based on 'time' column
    time_value = session_id.replace('ses-', '')
    subject_data = subject_data[subject_data['time'] == time_value]
    
    if subject_data.empty:
        logger.warning(f"No data found for {subject_id} {session_id}")
        return None
    
    # Extract volumetric columns
    volume_data = []
    
    # Process each brain region
    for region in BRAIN_REGIONS:
        for hemisphere in HEMISPHERES:
            # Column names in format: "A8m_L Vgm", "A8m_L Vwm", "A8m_L Vcsf"
            vgm_col = f"{region}_{hemisphere} Vgm"
            vwm_col = f"{region}_{hemisphere} Vwm"
            vcsf_col = f"{region}_{hemisphere} Vcsf"
            
            # Check if all tissue type columns exist
            if all(col in subject_data.columns for col in [vgm_col, vwm_col, vcsf_col]):
                try:
                    volume_data.append({
                        'region': region,
                        'hemisphere': hemisphere,
                        'gray_matter_volume': float(subject_data[vgm_col].values[0]),
                        'white_matter_volume': float(subject_data[vwm_col].values[0]),
                        'csf_volume': float(subject_data[vcsf_col].values[0])
                    })
                except (IndexError, ValueError) as e:
                    logger.warning(f"Failed to extract data for {region}_{hemisphere}: {e}")
                    continue
            else:
                logger.warning(f"Missing columns for region {region}_{hemisphere}")
    
    if not volume_data:
        logger.error(f"No valid volumetric data extracted for {subject_id} {session_id}")
        return None
    
    # Create BIDS derivative dataframe
    bids_df = pd.DataFrame(volume_data)
    
    # Create output directory
    try:
        anat_dir = create_directory_structure(output_dir, subject_id, session_id)
    except OSError:
        return None
    
    # Save as TSV
    output_file = anat_dir / f"{subject_id}_{session_id}_space-BNA_volumes.tsv"
    try:
        bids_df.to_csv(output_file, sep='\t', index=False)
        print(f"✓ Created: {output_file}")
        return str(output_file)
    except Exception as e:
        logger.error(f"Failed to write TSV file: {e}")
        return None

print("✓ CSV to BIDS conversion function defined")

✓ CSV to BIDS conversion function defined


In [44]:
def create_sidecar_json(output_dir, subject_id, session_id):
    """
    Create BIDS sidecar JSON file with column descriptions.
    
    Sidecar files provide metadata about the data columns,
    making the dataset self-documenting.
    
    Parameters:
    -----------
    output_dir : str
        Output directory for BIDS dataset
    subject_id : str
        Subject identifier
    session_id : str
        Session identifier
        
    Returns:
    --------
    bool
        True if successful, False otherwise
    """
    sidecar = {
        "region": {
            "LongName": "Brain Region",
            "Description": "Brainnetome Atlas (BNA) region label"
        },
        "hemisphere": {
            "LongName": "Hemisphere",
            "Description": "Brain hemisphere",
            "Levels": {
                "L": "Left hemisphere",
                "R": "Right hemisphere"
            }
        },
        "gray_matter_volume": {
            "LongName": "Gray Matter Volume",
            "Description": "Volume of gray matter tissue in the region",
            "Units": "mm^3"
        },
        "white_matter_volume": {
            "LongName": "White Matter Volume",
            "Description": "Volume of white matter tissue in the region",
            "Units": "mm^3"
        },
        "csf_volume": {
            "LongName": "Cerebrospinal Fluid Volume",
            "Description": "Volume of CSF in the region",
            "Units": "mm^3"
        }
    }
    
    anat_dir = Path(output_dir) / 'derivatives' / 'bna_volumes' / subject_id / session_id / 'anat'
    json_file = anat_dir / f"{subject_id}_{session_id}_space-BNA_volumes.json"
    
    try:
        with open(json_file, 'w') as f:
            json.dump(sidecar, f, indent=4)
        print(f"✓ Created: {json_file}")
        return True
    except Exception as e:
        logger.error(f"Failed to write JSON file: {e}")
        return False

print("✓ Sidecar JSON creation function defined")

✓ Sidecar JSON creation function defined


In [45]:
def create_participants_files(questionnaire_csv, volumetric_csv, output_dir):
    """
    Create BIDS participants.tsv and participants.json files.
    
    These files describe the demographics and characteristics
    of all participants in the dataset.
    
    Parameters:
    -----------
    questionnaire_csv : str
        Path to questionnaire metadata CSV
    volumetric_csv : str
        Path to volumetric data CSV (contains group assignment)
    output_dir : str
        Output directory for BIDS dataset
        
    Returns:
    --------
    DataFrame or None
        Participants dataframe if successful, None otherwise
    """
    # Validate input files
    questionnaire_path = Path(questionnaire_csv)
    volumetric_path = Path(volumetric_csv)
    
    if not questionnaire_path.exists():
        logger.error(f"Questionnaire file not found: {questionnaire_csv}")
        return None
    
    if not volumetric_path.exists():
        logger.error(f"Volumetric data file not found: {volumetric_csv}")
        return None
    
    try:
        # Read questionnaire data
        quest_df = pd.read_csv(questionnaire_csv)
        
        # Read volumetric data for group assignment
        vol_df = pd.read_csv(volumetric_csv)
        
        # Validate required columns
        required_quest_cols = ['subject num', 'age', 'is_female', 'is_right_handed', 'additional_languages']
        missing_quest_cols = [col for col in required_quest_cols if col not in quest_df.columns]
        if missing_quest_cols:
            logger.error(f"Missing questionnaire columns: {missing_quest_cols}")
            return None
        
        if 'group' not in vol_df.columns:
            logger.error("Missing 'group' column in volumetric data")
            return None
        
        # Get unique subject-group mapping from volumetric data
        subject_groups = vol_df[['subject num', 'group']].drop_duplicates()
        
        # Merge questionnaire data with group information
        merged_df = pd.merge(quest_df, subject_groups, on='subject num', how='left')
        
        # Create participants dataframe
        participants = pd.DataFrame({
            'participant_id': merged_df['subject num'].apply(lambda x: f'sub-{x}'),
            'age': merged_df['age'],
            'sex': merged_df['is_female'].map({0: 'M', 1: 'F'}),
            'group': merged_df['group'],
            'handedness': merged_df['is_right_handed'].map({0: 'L', 1: 'R'}),
            'additional_languages': merged_df['additional_languages']
        })
        
        # Save participants.tsv
        participants_file = Path(output_dir) / 'participants.tsv'
        participants.to_csv(participants_file, sep='\t', index=False)
        print(f"✓ Created: {participants_file}")
        
        # Create participants.json
        participants_json = {
            "age": {
                "Description": "Age of participant at time of data collection",
                "Units": "years"
            },
            "sex": {
                "Description": "Biological sex of participant",
                "Levels": {
                    "M": "male",
                    "F": "female"
                }
            },
            "group": {
                "Description": "Experimental group assignment",
                "Levels": {
                    "control": "Control group participants",
                    "patient": "Patient group participants"
                }
            },
            "handedness": {
                "Description": "Dominant hand of participant",
                "Levels": {
                    "R": "right-handed",
                    "L": "left-handed"
                }
            },
            "additional_languages": {
                "Description": "Additional languages spoken by participant at basic conversational level or above",
                "LongName": "Additional Languages"
            }
        }
        
        json_file = Path(output_dir) / 'participants.json'
        with open(json_file, 'w') as f:
            json.dump(participants_json, f, indent=4)
        print(f"✓ Created: {json_file}")
        
        return participants
        
    except Exception as e:
        logger.error(f"Failed to create participants files: {e}")
        return None

print("✓ Participants file creation function defined")

✓ Participants file creation function defined


## Step 4: Configure the Pipeline

Set up the configuration for our BIDS conversion.

In [46]:
# Configuration
input_data_csv = 'Data/sample_data.csv'
input_questionnaire_csv = 'Data/questionnaire_metadata.csv'
output_dir = '.'

# Subject and session information
subjects = [
    {'id': 'sub-1001', 'sessions': ['ses-pre']},
    {'id': 'sub-1002', 'sessions': ['ses-pre']},
    {'id': 'sub-1003', 'sessions': ['ses-post']}
]

print("Configuration:")
print(f"  Input data: {input_data_csv}")
print(f"  Input metadata: {input_questionnaire_csv}")
print(f"  Output directory: {output_dir}")
print(f"  Number of subjects: {len(subjects)}")

Configuration:
  Input data: Data/sample_data.csv
  Input metadata: Data/questionnaire_metadata.csv
  Output directory: .
  Number of subjects: 3


## Step 5: Run the Conversion Pipeline

Now let's execute the actual conversion process.

In [47]:
print("=" * 60)
print("BIDS Conversion Pipeline - STARTING")
print("=" * 60)

print("\n📋 Step 1: Creating participants files...")
participants_df = create_participants_files(input_questionnaire_csv, input_data_csv, output_dir)

if participants_df is not None:
    print("\n👤 Participants Summary:")
    display(participants_df)
else:
    print("❌ Failed to create participants files")

BIDS Conversion Pipeline - STARTING

📋 Step 1: Creating participants files...
✓ Created: participants.tsv
✓ Created: participants.json

👤 Participants Summary:


,participant_id,age,sex,group,handedness,additional_languages
0,sub-1001,28,M,control,R,"English, Hebrew"
1,sub-1002,34,F,patient,R,"English, Russian"
2,sub-1003,25,M,control,R,English


In [48]:
print("\n🧠 Step 2: Converting volumetric data to BIDS format...\n")

conversion_summary = []

for subject in subjects:
    subject_id = subject['id']
    for session_id in subject['sessions']:
        print(f"\nProcessing {subject_id} {session_id}...")
        
        # Convert data to BIDS TSV
        tsv_file = convert_csv_to_bids_tsv(
            input_data_csv, 
            output_dir, 
            subject_id, 
            session_id
        )
        
        # Create sidecar JSON
        if tsv_file:
            if create_sidecar_json(output_dir, subject_id, session_id):
                conversion_summary.append({
                    'subject': subject_id,
                    'session': session_id,
                    'status': '✓ Success'
                })
            else:
                conversion_summary.append({
                    'subject': subject_id,
                    'session': session_id,
                    'status': '✗ JSON failed'
                })
        else:
            conversion_summary.append({
                'subject': subject_id,
                'session': session_id,
                'status': '✗ No data'
            })

print("\n" + "=" * 60)
print("Conversion Summary:")
print("=" * 60)
summary_df = pd.DataFrame(conversion_summary)
display(summary_df)


🧠 Step 2: Converting volumetric data to BIDS format...


Processing sub-1001 ses-pre...
✓ Created: derivatives\bna_volumes\sub-1001\ses-pre\anat\sub-1001_ses-pre_space-BNA_volumes.tsv
✓ Created: derivatives\bna_volumes\sub-1001\ses-pre\anat\sub-1001_ses-pre_space-BNA_volumes.json

Processing sub-1002 ses-pre...
✓ Created: derivatives\bna_volumes\sub-1002\ses-pre\anat\sub-1002_ses-pre_space-BNA_volumes.tsv
✓ Created: derivatives\bna_volumes\sub-1002\ses-pre\anat\sub-1002_ses-pre_space-BNA_volumes.json

Processing sub-1003 ses-post...
✓ Created: derivatives\bna_volumes\sub-1003\ses-post\anat\sub-1003_ses-post_space-BNA_volumes.tsv
✓ Created: derivatives\bna_volumes\sub-1003\ses-post\anat\sub-1003_ses-post_space-BNA_volumes.json

Conversion Summary:


,subject,session,status
0,sub-1001,ses-pre,✓ Success
1,sub-1002,ses-pre,✓ Success
2,sub-1003,ses-post,✓ Success


## Step 6: Verify BIDS Structure

Let's check that our BIDS dataset structure is correct.

In [38]:
print("📁 BIDS Directory Structure:")
print("\nExpected structure:")
print("""
OpenScience/
├── dataset_description.json
├── participants.tsv
├── participants.json
├── README
├── CHANGES
└── derivatives/
    └── bna_volumes/
        ├── dataset_description.json
        └── sub-<ID>/
            └── ses-<session>/
                └── anat/
                    ├── sub-<ID>_ses-<session>_space-BNA_volumes.tsv
                    └── sub-<ID>_ses-<session>_space-BNA_volumes.json
""")

# Check for required files
required_files = [
    'dataset_description.json',
    'participants.tsv',
    'participants.json',
    'README',
    'CHANGES'
]

print("\nChecking required BIDS files:")
for file in required_files:
    file_path = Path(output_dir) / file
    exists = "✓" if file_path.exists() else "✗"
    print(f"{exists} {file}")

📁 BIDS Directory Structure:

Expected structure:

OpenScience/
├── dataset_description.json
├── participants.tsv
├── participants.json
├── README
├── CHANGES
└── derivatives/
    └── bna_volumes/
        ├── dataset_description.json
        └── sub-<ID>/
            └── ses-<session>/
                └── anat/
                    ├── sub-<ID>_ses-<session>_space-BNA_volumes.tsv
                    └── sub-<ID>_ses-<session>_space-BNA_volumes.json


Checking required BIDS files:
✓ dataset_description.json
✓ participants.tsv
✓ participants.json
✓ README
✓ CHANGES


## Step 7: Preview Sample Data

Let's look at a sample of the converted BIDS data.

In [25]:
# Load and display a sample TSV file
sample_tsv = Path(output_dir) / 'derivatives' / 'bna_volumes' / 'sub-1001' / 'ses-pre' / 'anat' / 'sub-1001_ses-pre_space-BNA_volumes.tsv'

if sample_tsv.exists():
    print("📊 Sample BIDS-formatted data (sub-1001, ses-pre):\n")
    sample_data = pd.read_csv(sample_tsv, sep='\t')
    display(sample_data)
    
    print("\n📈 Summary statistics:")
    display(sample_data.describe())
else:
    print("⚠ Sample file not found")

📊 Sample BIDS-formatted data (sub-1001, ses-pre):



,region,hemisphere,gray_matter_volume,white_matter_volume,csf_volume
0,A8m,L,2.82,0.98,0.54
1,A8m,R,3.71,1.04,0.82
2,A9l,L,2.78,0.62,1.21
3,A9l,R,3.13,0.87,1.40
4,A6dl,L,2.43,1.18,0.78
5,A6dl,R,2.49,0.88,1.11
6,A10m,L,4.19,1.72,1.04
7,A10m,R,3.91,1.25,1.15
8,mAmyg,L,1.45,0.68,0.30
9,mAmyg,R,1.26,0.57,0.25



📈 Summary statistics:


,gray_matter_volume,white_matter_volume,csf_volume
count,16.000000,16.000000,16.000000
mean,2.195625,0.788125,0.650000
std,1.126670,0.375397,0.423745
min,0.890000,0.320000,0.080000
25%,1.267500,0.537500,0.292500
50%,1.940000,0.660000,0.515000
75%,2.897500,0.995000,1.057500
max,4.190000,1.720000,1.400000


## Step 8: Next Steps and Validation

### What to do next:

1. **Validate BIDS structure**
   ```bash
   bids-validator .
   ```
   Install the validator: `npm install -g bids-validator`

2. **Review generated files**
   - Check `README` for completeness
   - Verify `dataset_description.json` metadata
   - Ensure all subjects/sessions are included

3. **Upload to OpenNeuro**
   - Create account at https://openneuro.org
   - Upload dataset
   - Complete metadata forms
   - Publish and obtain DOI

4. **Update documentation**
   - Add publication DOI to `dataset_description.json`
   - Update `CHANGES` file with version information

### FAIR Principles Achieved:

- ✅ **Findable:** Structured format with metadata, will have DOI from OpenNeuro
- ✅ **Accessible:** Open repository, standard format
- ✅ **Interoperable:** BIDS standard ensures compatibility with analysis tools
- ✅ **Reusable:** Comprehensive documentation and metadata

## Conclusion

This notebook has demonstrated a complete pipeline for converting neuroimaging volumetric data from CSV format to BIDS-compliant derivatives. The resulting dataset is:

- **Standardized:** Follows BIDS specification v1.8.0
- **Well-documented:** Includes comprehensive metadata
- **Reproducible:** This pipeline can be reused for future datasets
- **Shareable:** Ready for upload to OpenNeuro or similar repositories
- **Robust:** Includes error handling and validation

**Key Improvements Made:**
- ✅ Proper group assignment from source data
- ✅ Complete error handling and input validation
- ✅ Constants defined for brain regions and tissue types
- ✅ Logging for better tracking and debugging
- ✅ Comprehensive docstrings with examples

---

**Authors:** Gal Gvili & Tamir Rahamim  
**Date:** January 2026  
**Course:** 1501105101 - Theory and Practice: Opening Neuroscience